In [1]:
import pandas as pd
import sys
from qiskit import QuantumCircuit
import pylatexenc

In [3]:
circuit_path = "/Users/andrewweiland/UCCS_REU/quantum-optimizer-orchestration/benchmarks/ai_transpile/qasm/efficient_su2_8.qasm"

circuit = QuantumCircuit.from_qasm_file(circuit_path)
print(circuit)
print("Depth:", circuit.depth())
print("2Q gates:", circuit.num_nonlocal_gates())
print("Total gates:", circuit.size())

     ┌─────────┐┌─────────┐                                                  »
q_0: ┤ Ry(0.1) ├┤ Rz(4.7) ├──■────■─────────■─────────■──────────────■───────»
     ├─────────┤├─────────┤┌─┴─┐  │         │         │              │       »
q_1: ┤ Ry(1.2) ├┤ Rz(4.8) ├┤ X ├──┼────■────┼────■────┼─────────■────┼───────»
     ├─────────┤├─────────┤└───┘┌─┴─┐┌─┴─┐  │    │    │         │    │       »
q_2: ┤ Ry(2.3) ├┤ Rz(0.2) ├─────┤ X ├┤ X ├──┼────┼────┼────■────┼────┼────■──»
     ├─────────┤├─────────┤     └───┘└───┘┌─┴─┐┌─┴─┐  │  ┌─┴─┐  │    │    │  »
q_3: ┤ Ry(3.4) ├┤ Rz(0.3) ├───────────────┤ X ├┤ X ├──┼──┤ X ├──┼────┼────┼──»
     ├─────────┤├─────────┤               └───┘└───┘┌─┴─┐└───┘┌─┴─┐  │  ┌─┴─┐»
q_4: ┤ Ry(4.3) ├┤ Rz(0.4) ├─────────────────────────┤ X ├─────┤ X ├──┼──┤ X ├»
     ├─────────┤├─────────┤                         └───┘     └───┘┌─┴─┐└───┘»
q_5: ┤ Ry(4.4) ├┤ Rz(0.5) ├────────────────────────────────────────┤ X ├─────»
     ├─────────┤├─────────┤                         

In [4]:
basis_gates=["id", "rz", "sx", "x", "cx"]


In [5]:
#qiskit standard
from qiskit import transpile

qiskit_transpiled = transpile(circuit,
                            basis_gates=["id", "rz", "sx", "x", "cx"],
                            optimization_level=3,
                            backend=None,
                            coupling_map=None
                            )

print(qiskit_transpiled)
print("Depth:", qiskit_transpiled.depth())
print("2Q gates:", qiskit_transpiled.num_nonlocal_gates())
print("Total gates:", qiskit_transpiled.size())

     ┌────┐┌─────────────┐ ┌────┐ ┌────────────┐                              »
q_0: ┤ √X ├┤ Rz(-3.0416) ├─┤ √X ├─┤ Rz(1.5584) ├──■────■─────────■─────────■──»
     ├────┤├─────────────┤ ├────┤ ├────────────┤┌─┴─┐  │         │         │  »
q_1: ┤ √X ├┤ Rz(-1.9416) ├─┤ √X ├─┤ Rz(1.6584) ├┤ X ├──┼────■────┼────■────┼──»
     ├────┤├─────────────┴┐├────┤┌┴────────────┤└───┘┌─┴─┐┌─┴─┐  │    │    │  »
q_2: ┤ √X ├┤ Rz(-0.84159) ├┤ √X ├┤ Rz(-2.9416) ├─────┤ X ├┤ X ├──┼────┼────┼──»
     ├────┤├─────────────┬┘├────┤├─────────────┤     └───┘└───┘┌─┴─┐┌─┴─┐  │  »
q_3: ┤ √X ├┤ Rz(0.25841) ├─┤ √X ├┤ Rz(-2.8416) ├───────────────┤ X ├┤ X ├──┼──»
     ├────┤└┬────────────┤ ├────┤├─────────────┤               └───┘└───┘┌─┴─┐»
q_4: ┤ √X ├─┤ Rz(1.1584) ├─┤ √X ├┤ Rz(-2.7416) ├─────────────────────────┤ X ├»
     ├────┤ ├────────────┤ ├────┤├─────────────┤                         └───┘»
q_5: ┤ √X ├─┤ Rz(1.2584) ├─┤ √X ├┤ Rz(-2.6416) ├──────────────────────────────»
     ├────┤ ├────────────┤ ├────┤├──────

In [7]:
#qiskit ai
from qiskit.transpiler import CouplingMap
from qiskit.transpiler import CouplingMap, PassManager
from qiskit.transpiler.passes import (
    SabreLayout,
    CollectLinearFunctions,
    Optimize1qGates,
)

from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit_ibm_transpiler.ai.routing import AIRouting


def fully_connected_coupling_map(n):
    edges = [(i, j) for i in range(n) for j in range(n) if i != j]
    return CouplingMap(edges)



circuit_ibm_basis = transpile(
    circuit,
    basis_gates=basis_gates,
    optimization_level=0,
    coupling_map=None,
    backend=None,
)

coupling_map = fully_connected_coupling_map(circuit.num_qubits)

collect_lfs = CollectLinearFunctions(
    do_commutative_analysis=True,
    min_block_size=0,
    max_block_size=2,
    collect_from_back=False,
)

ai_lf_synth = AILinearFunctionSynthesis(
    coupling_map=list(coupling_map.get_edges()),
    replace_only_if_better=True,
)

pass_manager = PassManager(
    [
        collect_lfs,
        ai_lf_synth,
        Optimize1qGates(),
    ]
)
qiskit_ai_transpiled = pass_manager.run(circuit_ibm_basis)


qiskit_ai_transpiled = transpile(
    qiskit_ai_transpiled,
    basis_gates=["id", "rz", "sx", "x", "cx"],
    optimization_level=0
)

qiskit_ai_transpiled = pass_manager.run(circuit)

qiskit_ai_transpiled = transpile(
    qiskit_ai_transpiled,
    basis_gates=basis_gates,
    optimization_level=0,
    coupling_map=None,
    backend=None,
)

print("Depth:", qiskit_ai_transpiled.depth())
print("2Q gates:", qiskit_ai_transpiled.num_nonlocal_gates())
print("Total gates:", qiskit_ai_transpiled.size())

INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Linear function AI synthesis on local mode
/opt/anaconda3/envs/circuits/lib/python3.11/site-packages/networkx/algorithms/graph_hashing.py:211: UserWarning: The hashes produced for graphs without node or edge attributes changed in v3.5 due to a bugfix (see documentation).
  node_labels = _init_node_labels(G, edge_attr, node_attr)
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Linear function AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Linear function AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Linear function AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Linear function AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Linear function AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Linear functi

Depth: 39
2Q gates: 56
Total gates: 200


In [ ]:
#wisq standard
from wisq import optimize as wisq_optimize

wisq_transpiled = wisq_optimize(
                    input_path=circuit_path, 
                    output_path="optimized_circuits/wisq_standard.qasm",
                    target_gateset="IBMN",
                    timeout=300,
                    optimization_objective= "TWO_Q"
                    )

In [41]:

out_path = "optimized_circuits/wisq"

wisq_circuit = QuantumCircuit.from_qasm_file(out_path)

print("Depth:", wisq_circuit.depth())
print("2Q gates:", wisq_circuit.num_nonlocal_gates())
print("Total gates:", wisq_circuit.size())

Depth: 36
2Q gates: 56
Total gates: 176


In [42]:
#wisq bqskit
wisq_optimize(
            input_path=circuit_path, 
            output_path="optimized_circuits/wisq_bqskit.qasm",
            target_gateset="IBMN",
            timeout=300,
            optimization_objective= "TWO_Q",
            approximation_epsilon=1e-10  #Enables BQSKit resynthesis
            )

⠏     Starting resynthesis server...0m
⠋     Optimizing  (timeout: 300s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0:00:04

/opt/anaconda3/envs/circuits/lib/python3.11/site-packages/bqskit/passes/synthesis/leap.py:358: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  m, y_int, _, _, _ = linregress(best_layers, best_dists)


⠙     Optimizing  (timeout: 300s) ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0:00:07

/opt/anaconda3/envs/circuits/lib/python3.11/site-packages/bqskit/passes/synthesis/leap.py:358: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  m, y_int, _, _, _ = linregress(best_layers, best_dists)


⠸     Optimizing  (timeout: 300s) ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0:00:10

...

⠦     Optimizing  (timeout: 300s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺ 0:04:5104:50

/opt/anaconda3/envs/circuits/lib/python3.11/site-packages/bqskit/passes/synthesis/leap.py:358: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  m, y_int, _, _, _ = linregress(best_layers, best_dists)


⠴     Optimizing  (timeout: 300s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 0:04:57

/opt/anaconda3/envs/circuits/lib/python3.11/site-packages/bqskit/passes/synthesis/leap.py:358: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  m, y_int, _, _, _ = linregress(best_layers, best_dists)


      Optimizing  (timeout: 300s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0:05:00m 0:05:00


In [43]:
wisq_bqskit = QuantumCircuit.from_qasm_file("optimized_circuits/wisq_bqskit.qasm")

print("Depth:", wisq_bqskit.depth())
print("2Q gates:", wisq_bqskit.num_nonlocal_gates())
print("Total gates:", wisq_bqskit.size())

Depth: 48
2Q gates: 24
Total gates: 120


In [13]:
from bqskit import Circuit, compile

bqskit_circuit = Circuit.from_file(circuit_path)
compiled = compile(bqskit_circuit)
compiled.save("optimized_circuits/optimized_bqskit.qasm")

qc = QuantumCircuit.from_qasm_file("optimized_circuits/optimized_bqskit.qasm")
bqskit_transpiled = transpile(
    qc,
    basis_gates=["id", "rz", "sx", "x", "cx"],
    optimization_level=0,
)


print("Depth:", bqskit_transpiled.depth())
print("2Q gates:", bqskit_transpiled.num_nonlocal_gates())
print("Total gates:", bqskit_transpiled.size())

Depth: 36
2Q gates: 56
Total gates: 176
